# AprovaEdu Analytics — ETL e Tratamento de Dados

Este notebook documenta, passo a passo, o diagnóstico dos dados brutos e as
decisões de tratamento aplicadas antes da análise. A implementação reutilizável
está em `src/etl.py`; aqui reproduzimos o mesmo pipeline célula a célula,
mostrando **antes/depois** de cada tratamento, para deixar as decisões
transparentes para quem avaliar.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
import pandas as pd
import etl

pd.set_option("display.max_columns", 50)


## 1. Diagnóstico dos dados brutos

As 9 bases fornecidas (`data/raw/`) vêm de fontes internas diferentes e chegam
com inconsistências típicas de integração manual: variação de
maiúsculas/minúsculas, acentuação, abreviações, formatos de data distintos e
alguns valores fora de faixa. Alguns exemplos:

In [2]:
raw_estudantes = etl.load_raw("estudantes")
print("cidade (bruto):", sorted(raw_estudantes["cidade"].dropna().unique()))
print("escola_origem (bruto):", sorted(raw_estudantes["escola_origem"].dropna().unique()))


cidade (bruto): ['Aquiraz', 'Caucaia', 'Crato', 'Eusébio', 'FORTALEZA', 'Fortaleza', 'Horizonte', 'Itapipoca', 'Juazeiro do Norte', 'Maracanau', 'Maracanaú', 'Pacatuba', 'Sobral', 'fortaleza']
escola_origem (bruto): ['Federal', 'Não informado', 'Privada', 'Publica', 'Pública', 'privada']


In [3]:
raw_matriculas = etl.load_raw("matriculas")
print("materia_declarada (bruto):", sorted(raw_matriculas["materia_declarada"].dropna().unique()))
print("status_matricula (bruto):", sorted(raw_matriculas["status_matricula"].dropna().unique()))


materia_declarada (bruto): ['BIOLOGIA', 'Biologia', 'FILOSOFIA', 'Filosofia', 'Fisica', 'FÍSICA', 'Física', 'GEOGRAFIA', 'Geografia', 'Historia', 'História', 'Ingles', 'Inglês', 'MATEMÁTICA', 'Mat.', 'Matematica', 'Matemática', 'PORTUGUÊS', 'Portugues', 'Português', 'QUÍMICA', 'Quimica', 'Química', 'REDAÇÃO', 'Redacao', 'Redação', 'SOCIOLOGIA', 'Sociologia']
status_matricula (bruto): ['Ativa', 'Cancelada', 'Concluída', 'Trancada', 'concluida']


In [4]:
raw_resultados = etl.load_raw("resultados_simulados")
print("Formatos de data distintos em inicio_simulado (amostra):")
print(raw_resultados["inicio_simulado"].dropna().sample(8, random_state=1).tolist())
print("\nValores de nota fora da faixa 0-100:", ((raw_resultados["nota"] < 0) | (raw_resultados["nota"] > 100)).sum())


Formatos de data distintos em inicio_simulado (amostra):


['2022-07-10 15:45', '2025/06/22 13:30', '2021-09-28 17:30', '2025-04-01 12:30', '2024-06-15 10:15', '2023-10-10 19:45', '2025-05-08 09:30', '2024-11-10 14:30']

Valores de nota fora da faixa 0-100: 2101


## 2. Decisões de tratamento

As regras abaixo foram aplicadas de forma consistente em todas as tabelas
(implementadas em `src/etl.py`):

1. **Padronização de categorias**: cada coluna categórica (cidade, matéria,
   status de presença/matrícula/simulado, dispositivo, modalidade, etc.) é
   normalizada removendo acentuação/caixa para achar a chave e mapeada para
   um valor canônico único (ex.: `"presente"`, `"Presente"`, `"PRESENTE"` →
   `"Presente"`). Abreviações conhecidas (ex.: `"Mat."` → `"Matemática"`)
   também são tratadas.
2. **Valores ausentes em categorias**: preenchidos com `"Não informado"`
   (ou `"Não registrado"` para presença, para não confundir com a categoria
   `"Ausente"`, que representa falta efetiva).
3. **Datas em múltiplos formatos**: a base mistura `YYYY-MM-DD`,
   `YYYY/MM/DD`, `DD/MM/YYYY` e `MM-DD-YYYY` (com e sem hora). Um parser
   tolerante (`parse_date_flex`) identifica o padrão pela posição do ano e
   pelo separador e converte tudo para `datetime`.
4. **CPF fictício**: normalizado para apenas dígitos. Duplicatas de CPF
   fictício **não** foram tratadas como aluno duplicado, pois `aluno_id` é a
   chave primária real da base (sem duplicatas e sem chaves estrangeiras
   órfãs em nenhuma outra tabela).
5. **Notas de simulado fora da faixa 0–100** (ex.: `-3.5`, `1005`): tratadas
   como erro de digitação/importação. Em vez de tentar adivinhar uma correção,
   o valor é anulado e sinalizado na coluna `nota_valida=False`, preservando a
   linha (o aluno realizou o simulado) mas excluindo o valor da média.
6. **Professor duplicado**: o registro `P_DUP_001` ("Diego Rocha Lima") é uma
   duplicata de `P004` (mesmo nome) sem nenhuma referência em ofertas, aulas
   ou simulados — foi removido do cadastro de professores.
7. **Bolsa (`bolsa_percentual`) ausente**: preenchida com `0` (assume-se
   matrícula sem bolsa registrada).

In [5]:
estudantes = etl.clean_estudantes()
professores = etl.clean_professores()
ofertas = etl.clean_ofertas()
matriculas = etl.clean_matriculas()
aulas = etl.clean_aulas()
presencas = etl.clean_presencas()
simulados = etl.clean_simulados()
resultados = etl.clean_resultados_simulados()
aprovacoes = etl.clean_aprovacoes()
print("OK - todas as tabelas tratadas")


OK - todas as tabelas tratadas


## 3. Verificação: antes → depois

In [6]:
print("cidade (tratado):", sorted(estudantes["cidade"].dropna().unique()))
print("escola_origem (tratado):", sorted(estudantes["escola_origem"].dropna().unique()))
print("materia_declarada (tratado):", sorted(matriculas["materia_declarada"].dropna().unique()))
print("status_matricula (tratado):", sorted(matriculas["status_matricula"].dropna().unique()))


cidade (tratado): ['Aquiraz', 'Caucaia', 'Crato', 'Eusébio', 'Fortaleza', 'Horizonte', 'Itapipoca', 'Juazeiro do Norte', 'Maracanaú', 'Pacatuba', 'Sobral']
escola_origem (tratado): ['Federal', 'Não informado', 'Privada', 'Pública']
materia_declarada (tratado): ['Biologia', 'Filosofia', 'Física', 'Geografia', 'História', 'Inglês', 'Matemática', 'Português', 'Química', 'Redação', 'Sociologia']
status_matricula (tratado): ['Ativa', 'Cancelada', 'Concluída', 'Não informado', 'Trancada']


In [7]:
print("professores: 35 brutos ->", professores.shape[0], "após remover duplicata")
print("\nnotas de simulado invalidadas (fora de 0-100):", (~resultados["nota_valida"]).sum(), "de", len(resultados))
print("datas nao reconhecidas (NaT) em inicio_simulado:", resultados["inicio_simulado"].isna().sum())


professores: 35 brutos -> 34 após remover duplicata

notas de simulado invalidadas (fora de 0-100): 3787 de 21510
datas nao reconhecidas (NaT) em inicio_simulado: 0


## 4. Base estruturada para análise (marts)

Além das 9 tabelas tratadas, geramos duas tabelas agregadas (*marts*) que
alimentam diretamente as perguntas obrigatórias:

- **`mart_aluno_ano`**: uma linha por (aluno, ano), com taxa de presença nas
  aulas registradas naquele ano e indicador binário de aprovação no
  vestibular naquele ano — usada na pergunta 2.
- **`mart_curso_materia`**: uma linha por (ano, matéria), com nota média em
  simulados e taxa de conclusão de matrícula — usada na pergunta 3.

In [8]:
mart_aluno_ano = etl.build_mart_aluno_ano(estudantes, matriculas, presencas, aulas, aprovacoes)
mart_curso_materia = etl.build_mart_curso_materia(ofertas, matriculas, resultados, simulados, aprovacoes)
mart_aluno_ano.head()


,aluno_id,ano,aulas_registradas,aulas_presente,taxa_presenca,aprovado,cidade,escola_origem
0,A00001,2023,76,65,0.8553,1,Fortaleza,Não informado
1,A00001,2024,58,52,0.8966,1,Fortaleza,Não informado
2,A00002,2025,67,61,0.9104,0,Crato,Federal
3,A00003,2022,88,78,0.8864,0,Horizonte,Privada
4,A00004,2021,68,58,0.8529,0,Juazeiro do Norte,Pública


In [9]:
mart_curso_materia.head()


,ano,materia,total_matriculas,matriculas_concluidas,taxa_conclusao,nota_media_simulados
0,2021,Biologia,109,73,0.6697,61.037634
1,2021,Filosofia,111,87,0.7838,61.030769
2,2021,Física,112,64,0.5714,60.987117
3,2021,Geografia,109,72,0.6606,60.974925
4,2021,História,103,78,0.7573,61.893785


## 5. Exportação

As tabelas tratadas e as marts são exportadas para `data/processed/` em CSV
e também consolidadas em um banco SQLite (`data/processed/aprovaedu.db`),
reutilizável por qualquer ferramenta de BI/SQL.

In [10]:
etl.main()


Validacao de integridade: OK (9 chaves primarias, 11 FKs, 11 dominios categoricos checados)


ETL concluido. Tabelas geradas:
  - estudantes: 812 linhas, 10 colunas
  - professores: 34 linhas, 10 colunas
  - ofertas_curso: 220 linhas, 13 colunas
  - matriculas: 9452 linhas, 10 colunas
  - aulas: 2418 linhas, 10 colunas
  - presencas_aulas: 74997 linhas, 6 colunas
  - simulados: 165 linhas, 11 colunas
  - resultados_simulados: 21510 linhas, 13 colunas
  - aprovacoes_vestibular: 354 linhas, 11 colunas
  - mart_aluno_ano: 1022 linhas, 8 colunas
  - mart_curso_materia: 55 linhas, 6 colunas

CSV tratados em: C:\Users\Mobi2buy\Desktop\Claude Code\aprovaedu-analytics\data\processed
Banco SQLite em: C:\Users\Mobi2buy\Desktop\Claude Code\aprovaedu-analytics\data\processed\aprovaedu.db
